# 01 — Polymarket BTC Data Collection and Cleaning

This notebook collects, classifies, cleans, and stores Bitcoin-related prediction market data from Polymarket.

The output of this notebook is a cleaned Polymarket dataset that will be used in the cross-market comparison with Kalshi and Deribit.

In [3]:
from importlib import reload
import config
reload(config)
from config import *

print("BASE_DIR:", BASE_DIR)
print("PROCESSED_DIR:", PROCESSED_DIR)

BASE_DIR: /Users/giannandreadestefano/TESI FINANCE/Thesis Project
PROCESSED_DIR: /Users/giannandreadestefano/TESI FINANCE/Thesis Project/data/processed


In [5]:
# ============================================================
# Imports and project configuration
# ============================================================

import os
import re
import json
import time
import requests
from importlib import reload

import numpy as np
import pandas as pd

import config
reload(config)
from config import *

print("Sample period:", SAMPLE_START, "to", SAMPLE_END)
print("Download data:", DOWNLOAD_DATA)
print("Raw data directory:", RAW_DIR)
print("Processed data directory:", PROCESSED_DIR)
print("Final data directory:", FINAL_DIR)

for folder in [RAW_DIR, PROCESSED_DIR, FINAL_DIR, FIGURES_DIR, TABLES_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

Sample period: 2024-01-01 to 2026-06-04
Download data: True
Raw data directory: /Users/giannandreadestefano/TESI FINANCE/Thesis Project/data/raw
Processed data directory: /Users/giannandreadestefano/TESI FINANCE/Thesis Project/data/processed
Final data directory: /Users/giannandreadestefano/TESI FINANCE/Thesis Project/data/final


In [7]:
print("Sample period:", SAMPLE_START, "to", SAMPLE_END)
print("Download data:", DOWNLOAD_DATA)
print("Raw data directory:", RAW_DIR)
print("Processed data directory:", PROCESSED_DIR)
print("Final data directory:", FINAL_DIR)

Sample period: 2024-01-01 to 2026-06-04
Download data: True
Raw data directory: /Users/giannandreadestefano/TESI FINANCE/Thesis Project/data/raw
Processed data directory: /Users/giannandreadestefano/TESI FINANCE/Thesis Project/data/processed
Final data directory: /Users/giannandreadestefano/TESI FINANCE/Thesis Project/data/final


In [9]:
# ============================================================
# Polymarket API settings
# ============================================================

GAMMA_EVENTS_URL = "https://gamma-api.polymarket.com/events"
CLOB_HISTORY_URL = "https://clob.polymarket.com/prices-history"

REQUEST_LIMIT = 100
MAX_OFFSET = 10_000
REQUEST_DELAY = 0.20

In [11]:
# ============================================================
# Polymarket API helper functions
# ============================================================

def fetch_events(end_date_min, end_date_max, limit=REQUEST_LIMIT, offset=0):
    """
    Fetches a batch of Polymarket events from the Gamma API.
    """
    params = {
        "limit": limit,
        "offset": offset,
        "closed": "true",
        "end_date_min": end_date_min,
        "end_date_max": end_date_max,
    }

    response = requests.get(GAMMA_EVENTS_URL, params=params, timeout=30)
    response.raise_for_status()

    return response.json()


def get_history(token_id):
    """
    Downloads historical YES-token prices from Polymarket CLOB.

    Returns a dataframe with datetime and price columns.
    """
    if pd.isna(token_id):
        return pd.DataFrame()

    params = {
        "market": str(token_id),
        "interval": "max",
        "fidelity": 1440
    }

    try:
        response = requests.get(CLOB_HISTORY_URL, params=params, timeout=30)
        response.raise_for_status()
        data = response.json()
    except Exception:
        return pd.DataFrame()

    history = data.get("history", [])

    if not history:
        return pd.DataFrame()

    df_history = pd.DataFrame(history)

    if "t" not in df_history.columns or "p" not in df_history.columns:
        return pd.DataFrame()

    df_history["datetime"] = pd.to_datetime(df_history["t"], unit="s", utc=True)
    df_history["price"] = pd.to_numeric(df_history["p"], errors="coerce")

    df_history = (
        df_history[["datetime", "price"]]
        .dropna()
        .sort_values("datetime")
        .reset_index(drop=True)
    )

    return df_history

In [13]:
# ============================================================
# Market classification
# ============================================================

def is_intraday_market(question):
    """
    Identifies intraday BTC markets, which are excluded because their payoff
    structure is not comparable with daily, weekly, or monthly markets.
    """
    q = str(question).lower()
    return bool(re.search(r'\b(?:at\s*)?\d{1,2}\s*(?:am|pm)\s*et\b', q))

EXCLUDE_KEYWORDS = [
    "up or down",
    "microstrategy",
    "mstr",
    "aum",
    "1 hour after",
    "halving",
    "volatility index",
    "hashprice",
    "by tomorrow",
    "by tonight",
    "by end of day",
    "by sunday",
    "by monday",
    "by tuesday",
    "by wednesday",
    "by thursday",
    "by friday",
    "ethbtc",
    "eth/btc",
    "ethereum",
    " eth ",
    "btc or eth",
    "or eth",
    "all-time high first",
    "by saturday",
]


def classify_bet(question):
    """
    Classifies BTC-related Polymarket questions into economically comparable
    prediction market types.
    """
    q = str(question).lower()

    if is_intraday_market(q):
        return None

    for keyword in EXCLUDE_KEYWORDS:
        if keyword in q:
            return None

    if "hit" in q and " or " in q:
        return None

    if "between" in q:
        return "range"

    if "above" in q:
        return "above"

    if "dip" in q or "below" in q:
        return "dip"

    if "reach" in q:
        return "reach"

    return None

In [15]:
# ============================================================
# Strike and token extraction
# ============================================================

def extract_strike(question):
    """
    Extracts strike prices from BTC-related Polymarket questions.

    Handles formats such as:
    - $70,000
    - $70k
    - 70k
    - 119.5k
    - 120500
    - between $100K and $102K
    """
    q = str(question).lower().replace(",", "")
    values = []

    # Values expressed in k, with or without dollar sign.
    for match in re.finditer(r'(?<![\d.])\$?(\d+(?:\.\d+)?)\s*k\b', q):
        values.append(float(match.group(1)) * 1000)

    # Explicit dollar values without k.
    for match in re.finditer(r'\$(\d{4,7}(?:\.\d+)?)\b', q):
        values.append(float(match.group(1)))
    
    # Plain BTC-like values without dollar sign or k.
    for match in re.finditer(r'(?<![\d.])(\d{5,7})(?![\d.])', q):
        values.append(float(match.group(1)))

    values = sorted(set(value for value in values if value >= 1000))

    if len(values) >= 2:
        return values[0], values[1]

    if len(values) == 1:
        return values[0], None

    return None, None


def get_yes_token(market):
    """
    Extracts the YES token id from a Polymarket market object.
    """
    tokens = market.get("clobTokenIds", "[]")

    if isinstance(tokens, str):
        try:
            tokens = json.loads(tokens)
        except json.JSONDecodeError:
            return None

    return tokens[0] if tokens else None

In [17]:
# ============================================================
# Quick validation of classification and strike extraction
# ============================================================

test_questions = [
    "Bitcoin above $100,000 on June 4?",
    "Will Bitcoin be between $100K and $102K on June 4?",
    "Will Bitcoin reach 119.5k on August 17?",
    "Will Bitcoin dip to 117.5k on August 17?",
    "Will Bitcoin reach 120500 on August 15?",
    "Bitcoin above $117K on August 17 at 12AM ET?",
    "Bitcoin above 70,000 on March 25, 9AM ET?",
    "Bitcoin above 70,000 on March 25, 8PM ET?",
    "Bitcoin up or down on June 4?",
]

for question in test_questions:
    bet_type = classify_bet(question)
    strike_lo, strike_hi = extract_strike(question)

    print(question)
    print(f"  type: {bet_type}, strike_lo: {strike_lo}, strike_hi: {strike_hi}")

Bitcoin above $100,000 on June 4?
  type: above, strike_lo: 100000.0, strike_hi: None
Will Bitcoin be between $100K and $102K on June 4?
  type: range, strike_lo: 100000.0, strike_hi: 102000.0
Will Bitcoin reach 119.5k on August 17?
  type: reach, strike_lo: 119500.0, strike_hi: None
Will Bitcoin dip to 117.5k on August 17?
  type: dip, strike_lo: 117500.0, strike_hi: None
Will Bitcoin reach 120500 on August 15?
  type: reach, strike_lo: 120500.0, strike_hi: None
Bitcoin above $117K on August 17 at 12AM ET?
  type: None, strike_lo: 117000.0, strike_hi: None
Bitcoin above 70,000 on March 25, 9AM ET?
  type: None, strike_lo: 70000.0, strike_hi: None
Bitcoin above 70,000 on March 25, 8PM ET?
  type: None, strike_lo: 70000.0, strike_hi: None
Bitcoin up or down on June 4?
  type: None, strike_lo: None, strike_hi: None


In [19]:
# ============================================================
# Event scanning windows
# ============================================================

windows = [
    ("2024-01-01", "2024-12-31"),

    ("2025-01-01", "2025-01-31"),
    ("2025-02-01", "2025-02-28"),
    ("2025-03-01", "2025-03-31"),
    ("2025-04-01", "2025-04-30"),
    ("2025-05-01", "2025-05-31"),
    ("2025-06-01", "2025-06-30"),
    ("2025-07-01", "2025-07-31"),

    ("2025-08-01", "2025-08-15"),
    ("2025-08-16", "2025-08-31"),
    ("2025-09-01", "2025-09-15"),
    ("2025-09-16", "2025-09-30"),
    ("2025-10-01", "2025-10-15"),
    ("2025-10-16", "2025-10-31"),

    ("2025-11-01", "2025-11-30"),
    ("2025-12-01", "2025-12-31"),

    ("2026-01-01", "2026-01-31"),
    ("2026-02-01", "2026-02-28"),
    ("2026-03-01", "2026-03-31"),
    ("2026-04-01", "2026-04-30"),
    ("2026-05-01", "2026-05-31"),
    ("2026-06-01", SAMPLE_END),
]

In [21]:
# ============================================================
# Metadata collection
# ============================================================

def scan_btc_events(end_date_min, end_date_max, max_offset=MAX_OFFSET, verbose=True):
    """
    Scans Polymarket events and extracts metadata for BTC-related markets.
    """
    metadata = []
    seen_market_ids = set()

    if verbose:
        print(f"Scanning events from {end_date_min} to {end_date_max}")

    for offset in range(0, max_offset + REQUEST_LIMIT, REQUEST_LIMIT):
        events = fetch_events(
            end_date_min=end_date_min,
            end_date_max=end_date_max,
            limit=REQUEST_LIMIT,
            offset=offset
        )

        if not events:
            break

        for event in events:
            event_id = event.get("id")
            markets = event.get("markets", [])

            for market in markets:
                question = market.get("question", "")

                if "bitcoin" not in question.lower() and "btc" not in question.lower():
                    continue

                bet_type = classify_bet(question)

                if bet_type is None:
                    continue

                market_id = market.get("id")

                if market_id in seen_market_ids:
                    continue

                seen_market_ids.add(market_id)

                strike_lo, strike_hi = extract_strike(question)
                yes_token = get_yes_token(market)

                metadata.append({
                    "event_id": event_id,
                    "market_id": market_id,
                    "question": question,
                    "bet_type": bet_type,
                    "start_date": market.get("startDate"),
                    "end_date": market.get("endDate"),
                    "closed": market.get("closed"),
                    "volume": market.get("volumeNum", 0),
                    "yes_token": yes_token,
                    "strike_lo": strike_lo,
                    "strike_hi": strike_hi,
                })

        if len(events) < REQUEST_LIMIT:
            break

        time.sleep(REQUEST_DELAY)

    return pd.DataFrame(metadata)

In [23]:
# ============================================================
# Load or download Polymarket BTC metadata
# ============================================================

meta_path = PROCESSED_DIR / "polymarket_btc_meta.csv"

if DOWNLOAD_POLYMARKET:
    print("Downloading Polymarket BTC metadata from scratch...")
    print(f"Sample window: {SAMPLE_START} to {SAMPLE_END}")
    print(f"Output file: {meta_path}\n")

    metadata_frames = []

    for start_date, end_date in windows:
        print(f"Scanning window: {start_date} -> {end_date}")
    
        df_window = scan_btc_events(
            end_date_min=start_date,
            end_date_max=end_date,
            max_offset=MAX_OFFSET,
            verbose=True
        )
    
        if not df_window.empty:
            df_window = df_window.dropna(axis=1, how="all")
            metadata_frames.append(df_window)
            print(f"  Markets found: {len(df_window):,}\n")
        else:
            print("  No BTC markets found in this window.\n")
    
        time.sleep(REQUEST_DELAY)

    if not metadata_frames:
        raise ValueError(
            "No Polymarket BTC metadata were downloaded. "
            "Check API settings, date windows, or market filters."
        )

    df_meta = (
        pd.concat(metadata_frames, ignore_index=True)
        .drop_duplicates(subset="market_id")
        .reset_index(drop=True)
    )

    # Basic cleaning before saving
    df_meta["start_date"] = pd.to_datetime(df_meta["start_date"], errors="coerce", utc=True)
    df_meta["end_date"] = pd.to_datetime(df_meta["end_date"], errors="coerce", utc=True)

    sample_start_ts = pd.Timestamp(SAMPLE_START, tz="UTC")
    sample_end_ts = pd.Timestamp(SAMPLE_END, tz="UTC") + pd.Timedelta(days=1) - pd.Timedelta(seconds=1)

    df_meta = df_meta[
        (df_meta["end_date"] >= sample_start_ts) &
        (df_meta["end_date"] <= sample_end_ts)
    ].copy()

    if USE_ONLY_CLOSED_MARKETS:
        df_meta = df_meta[df_meta["closed"] == True].copy()

    df_meta["strike_lo"] = pd.to_numeric(df_meta["strike_lo"], errors="coerce")
    df_meta["strike_hi"] = pd.to_numeric(df_meta["strike_hi"], errors="coerce")
    df_meta["volume"] = pd.to_numeric(df_meta["volume"], errors="coerce").fillna(0)

    df_meta = (
        df_meta
        .sort_values(["end_date", "bet_type", "strike_lo"])
        .reset_index(drop=True)
    )

    df_meta.to_csv(meta_path, index=False)

    print("=" * 70)
    print("Polymarket BTC metadata downloaded and saved successfully.")
    print(f"Saved to: {meta_path}")
    print(f"Total markets: {len(df_meta):,}")

else:
    if not meta_path.exists():
        raise FileNotFoundError(
            f"Metadata file not found: {meta_path}\n"
            "Set DOWNLOAD_DATA = True in config.py to download metadata from scratch."
        )

    df_meta = pd.read_csv(meta_path)

    df_meta["start_date"] = pd.to_datetime(df_meta["start_date"], errors="coerce", utc=True)
    df_meta["end_date"] = pd.to_datetime(df_meta["end_date"], errors="coerce", utc=True)

    print(f"Loaded existing Polymarket BTC metadata from: {meta_path}")

print("=" * 70)
print(f"Polymarket BTC markets loaded: {len(df_meta):,}")

if len(df_meta) > 0:
    print(f"Date range: {df_meta['end_date'].min()} -> {df_meta['end_date'].max()}")
    print("\nMarkets by bet type:")
    print(df_meta["bet_type"].value_counts().to_string())

    print("\nStrike coverage:")
    strike_coverage = (
        df_meta.groupby("bet_type")["strike_lo"]
        .apply(lambda x: x.notna().mean())
        .mul(100)
        .round(1)
    )
    print(strike_coverage.to_string())

Sample window: 2024-01-01 to 2026-06-04
Output file: /Users/giannandreadestefano/TESI FINANCE/Thesis Project/data/processed/polymarket_btc_meta.csv

Scanning window: 2024-01-01 -> 2024-12-31
Scanning events from 2024-01-01 to 2024-12-31
  Markets found: 105

Scanning window: 2025-01-01 -> 2025-01-31
Scanning events from 2025-01-01 to 2025-01-31
  Markets found: 4

Scanning window: 2025-02-01 -> 2025-02-28
Scanning events from 2025-02-01 to 2025-02-28
  Markets found: 9

Scanning window: 2025-03-01 -> 2025-03-31
Scanning events from 2025-03-01 to 2025-03-31
  Markets found: 24

Scanning window: 2025-04-01 -> 2025-04-30
Scanning events from 2025-04-01 to 2025-04-30
  Markets found: 24

Scanning window: 2025-05-01 -> 2025-05-31
Scanning events from 2025-05-01 to 2025-05-31
  Markets found: 41

Scanning window: 2025-06-01 -> 2025-06-30
Scanning events from 2025-06-01 to 2025-06-30
  Markets found: 32

Scanning window: 2025-07-01 -> 2025-07-31
Scanning events from 2025-07-01 to 2025-07-31
 

In [33]:
# ============================================================
# Metadata cleaning
# ============================================================

print("=" * 70)
print("Cleaning Polymarket BTC metadata")
print("=" * 70)

initial_markets = len(df_meta)

# Recompute classification and strikes using the final functions.
df_meta["bet_type"] = df_meta["question"].apply(classify_bet)

strike_values = df_meta["question"].apply(lambda q: pd.Series(extract_strike(q)))
strike_values.columns = ["strike_lo", "strike_hi"]
df_meta[["strike_lo", "strike_hi"]] = strike_values

# Normalize core columns.
df_meta["start_date"] = pd.to_datetime(df_meta["start_date"], errors="coerce", utc=True)
df_meta["end_date"] = pd.to_datetime(df_meta["end_date"], errors="coerce", utc=True)
df_meta["strike_lo"] = pd.to_numeric(df_meta["strike_lo"], errors="coerce")
df_meta["strike_hi"] = pd.to_numeric(df_meta["strike_hi"], errors="coerce")
df_meta["volume"] = pd.to_numeric(df_meta["volume"], errors="coerce").fillna(0)

# Store rows excluded by final filters.
invalid_bet_mask = df_meta["bet_type"].isna()
missing_strike_mask = df_meta["strike_lo"].isna()

df_meta_excluded = df_meta[invalid_bet_mask | missing_strike_mask].copy()

if not df_meta_excluded.empty:
    excluded_path = PROCESSED_DIR / "polymarket_btc_meta_excluded.csv"
    df_meta_excluded.to_csv(excluded_path, index=False)
    print(f"Excluded markets saved to: {excluded_path}")

# Keep final clean metadata.
df_meta = df_meta[
    df_meta["bet_type"].notna() &
    df_meta["strike_lo"].notna()
].copy()

df_meta = (
    df_meta
    .drop_duplicates(subset="market_id")
    .sort_values(["end_date", "bet_type", "strike_lo"])
    .reset_index(drop=True)
)

meta_path = PROCESSED_DIR / "polymarket_btc_meta.csv"
df_meta.to_csv(meta_path, index=False)

print(f"Initial markets: {initial_markets:,}")
print(f"Excluded markets: {len(df_meta_excluded):,}")
print(f"Final clean markets: {len(df_meta):,}")
print(f"Missing strike_lo: {df_meta['strike_lo'].isna().sum()}")
print(f"Saved to: {meta_path}")

Cleaning Polymarket BTC metadata
Initial markets: 3,714
Excluded markets: 0
Final clean markets: 3,714
Missing strike_lo: 0
Saved to: /Users/giannandreadestefano/TESI FINANCE/Thesis Project/data/processed/polymarket_btc_meta.csv


In [39]:
# ============================================================
# Metadata diagnostics
# ============================================================

print("=" * 70)
print("Polymarket BTC metadata diagnostics")
print("=" * 70)

meta_path = PROCESSED_DIR / "polymarket_btc_meta.csv"

print("Metadata file:", meta_path)
print("File exists:", meta_path.exists())

print("\nDataset size:")
print(f"Markets: {len(df_meta):,}")
print(f"Unique market_id: {df_meta['market_id'].nunique():,}")
print(f"Unique yes_token: {df_meta['yes_token'].nunique():,}")

print("\nMarkets by bet type:")
display(
    df_meta.groupby("bet_type")
    .agg(
        n_markets=("market_id", "nunique"),
        avg_volume=("volume", "mean"),
        strike_coverage=("strike_lo", lambda x: x.notna().mean())
    )
    .round(3)
)

print("\nMarkets by bet type and month:")
df_meta["end_month"] = df_meta["end_date"].dt.tz_convert(None).dt.to_period("M").astype(str)

display(
    df_meta.groupby(["bet_type", "end_month"])["market_id"]
    .nunique()
    .unstack(fill_value=0)
)

print("\nMissing strike:")
missing_strike = df_meta[df_meta["strike_lo"].isna()].copy()
print(f"Markets with missing strike: {len(missing_strike)}")

if len(missing_strike) > 0:
    display(missing_strike[["market_id", "bet_type", "question", "end_date"]].head(30))

print("\nResidual intraday markets:")
intraday_pattern = r"\b(?:at\s*)?\d{1,2}\s*(?:am|pm)\s*et\b"

intraday_left = df_meta[
    df_meta["question"].str.contains(
        intraday_pattern,
        case=False,
        regex=True,
        na=False
    )
].copy()

print(f"Intraday markets still included: {len(intraday_left)}")

if len(intraday_left) > 0:
    display(intraday_left[["market_id", "bet_type", "question", "end_date"]].head(30))

print("\nResidual short-horizon daily markets:")
daily_keywords = [
    "by tomorrow", "by tonight", "by end of day",
    "by sunday", "by monday", "by tuesday", "by wednesday",
    "by thursday", "by friday", "by saturday"
]

daily_pattern = "|".join(daily_keywords)

daily_left = df_meta[
    df_meta["question"].str.contains(
        daily_pattern,
        case=False,
        regex=True,
        na=False
    )
].copy()

print(f"Daily short-horizon markets still included: {len(daily_left)}")

if len(daily_left) > 0:
    display(daily_left[["market_id", "bet_type", "question", "end_date"]].head(30))

print("\nSample date range:")
print("Minimum end_date:", df_meta["end_date"].min())
print("Maximum end_date:", df_meta["end_date"].max())

sample_end_limit = pd.Timestamp(SAMPLE_END, tz="UTC") + pd.Timedelta(days=1)

outside_sample = df_meta[df_meta["end_date"] > sample_end_limit].copy()

print("\nMarkets outside sample:")
print(len(outside_sample))

if len(outside_sample) > 0:
    display(outside_sample[["market_id", "bet_type", "question", "end_date"]].head(30))

print("\nStrike range by bet type:")
for bet_type in sorted(df_meta["bet_type"].dropna().unique()):
    sub = df_meta[df_meta["bet_type"] == bet_type]
    print(
        f"{bet_type:>6}: "
        f"min={sub['strike_lo'].min():,.0f}, "
        f"max={sub['strike_lo'].max():,.0f}, "
        f"markets={len(sub):,}"
    )

df_meta = df_meta.drop(columns=["end_month"])

Polymarket BTC metadata diagnostics
Metadata file: /Users/giannandreadestefano/TESI FINANCE/Thesis Project/data/processed/polymarket_btc_meta.csv
File exists: True

Dataset size:
Markets: 3,714
Unique market_id: 3,714
Unique yes_token: 3,714

Markets by bet type:


,n_markets,avg_volume,strike_coverage
bet_type,,,
above,1579,372380.188,1.0
dip,387,767218.725,1.0
range,1337,127159.564,1.0
reach,411,1531510.347,1.0



Markets by bet type and month:


end_month,2024-03,2024-04,2024-05,2024-06,2024-07,2024-08,2024-09,2024-10,2024-11,2024-12,...,2025-09,2025-10,2025-11,2025-12,2026-01,2026-02,2026-03,2026-04,2026-05,2026-06
bet_type,,,,,,,,,,,,,,,,,,,,,
above,3,4,4,4,4,5,4,5,5,4,...,238,319,176,176,110,143,88,77,77,33
dip,0,0,4,0,5,5,3,3,0,0,...,37,37,32,36,13,24,25,33,35,42
range,0,0,0,0,0,0,0,0,0,0,...,185,261,144,153,90,117,72,54,54,27
reach,0,4,4,3,3,4,3,3,18,0,...,33,38,31,38,13,24,26,33,34,45



Missing strike:
Markets with missing strike: 0

Residual intraday markets:
Intraday markets still included: 0

Residual short-horizon daily markets:
Daily short-horizon markets still included: 0

Sample date range:
Minimum end_date: 2024-03-15 00:00:00+00:00
Maximum end_date: 2026-06-03 16:00:00+00:00

Markets outside sample:
0

Strike range by bet type:
 above: min=54,000, max=134,000, markets=1,579
   dip: min=20,000, max=122,000, markets=387
 range: min=56,000, max=132,000, markets=1,337
 reach: min=60,000, max=1,000,000, markets=411


In [41]:
# ============================================================
# Inspect extreme strike values
# ============================================================

extreme_strikes = df_meta[
    (df_meta["strike_lo"] < 40_000) |
    (df_meta["strike_lo"] > 200_000)
].copy()

print(f"Extreme strike markets: {len(extreme_strikes)}")

display(
    extreme_strikes[
        ["market_id", "bet_type", "question", "strike_lo", "strike_hi", "end_date", "volume"]
    ].sort_values(["strike_lo", "end_date"])
)

Extreme strike markets: 14


,market_id,bet_type,question,strike_lo,strike_hi,end_date,volume
2412,516873,dip,"Will Bitcoin dip to $20,000 by December 31, 2025?",20000.0,NaN,2025-12-31 12:00:00+00:00,4.003570e+06
3170,1473090,dip,"Will Bitcoin dip to $20,000 in March?",20000.0,NaN,2026-04-01 04:00:00+00:00,3.150011e+06
3367,1823788,dip,"Will Bitcoin dip to $20,000 in April?",20000.0,NaN,2026-05-01 04:00:00+00:00,1.818739e+06
3171,1473087,dip,"Will Bitcoin dip to $25,000 in March?",25000.0,NaN,2026-04-01 04:00:00+00:00,1.314840e+06
3368,1823787,dip,"Will Bitcoin dip to $25,000 in April?",25000.0,NaN,2026-05-01 04:00:00+00:00,3.961066e+05
3172,1473086,dip,"Will Bitcoin dip to $30,000 in March?",30000.0,NaN,2026-04-01 04:00:00+00:00,6.919259e+05
3369,1823786,dip,"Will Bitcoin dip to $30,000 in April?",30000.0,NaN,2026-05-01 04:00:00+00:00,8.325713e+05
3567,2132787,dip,"Will Bitcoin dip to $30,000 in May?",30000.0,NaN,2026-06-01 04:00:00+00:00,5.358629e+05
2959,1303410,dip,"Will Bitcoin dip to $35,000 in February?",35000.0,NaN,2026-03-01 05:00:00+00:00,4.546147e+06
3173,1473084,dip,"Will Bitcoin dip to $35,000 in March?",35000.0,NaN,2026-04-01 04:00:00+00:00,1.708798e+06


In [43]:
# ============================================================
# Strike distribution diagnostics
# ============================================================

display(
    df_meta.groupby("bet_type")["strike_lo"]
    .describe(percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99])
    .round(2)
)

,count,mean,std,min,1%,5%,25%,50%,75%,95%,99%,max
bet_type,,,,,,,,,,,,
above,1579.0,96730.84,18994.54,54000.0,58000.0,64000.0,82000.0,100000.0,112000.0,122000.0,128000.0,134000.0
dip,387.0,81687.34,23694.93,20000.0,25000.0,45000.0,65000.0,78000.0,104000.0,116000.0,118000.0,122000.0
range,1337.0,96822.74,17676.63,56000.0,60000.0,66000.0,82000.0,100000.0,112000.0,120000.0,126000.0,132000.0
reach,411.0,105985.40,51348.46,60000.0,65250.0,70000.0,80000.0,105000.0,120000.0,147500.0,200000.0,1000000.0


In [45]:
# ============================================================
# Main-sample flag based on economically comparable strike range
# ============================================================

MAIN_STRIKE_MIN = 40_000
MAIN_STRIKE_MAX = 200_000

df_meta["main_sample"] = (
    (df_meta["strike_lo"] >= MAIN_STRIKE_MIN) &
    (df_meta["strike_lo"] <= MAIN_STRIKE_MAX)
)

df_meta_main = df_meta[df_meta["main_sample"]].copy()

meta_path = PROCESSED_DIR / "polymarket_btc_meta.csv"
main_meta_path = PROCESSED_DIR / "polymarket_btc_meta_main.csv"

df_meta.to_csv(meta_path, index=False)
df_meta_main.to_csv(main_meta_path, index=False)

print("Full metadata sample:")
print(f"  Markets: {len(df_meta):,}")

print("\nMain metadata sample:")
print(f"  Markets: {len(df_meta_main):,}")
print(f"  Excluded markets: {len(df_meta) - len(df_meta_main):,}")

print("\nExcluded markets by bet type:")
print(
    df_meta.loc[~df_meta["main_sample"], "bet_type"]
    .value_counts()
    .to_string()
)

display(
    df_meta.loc[
        ~df_meta["main_sample"],
        ["market_id", "bet_type", "question", "strike_lo", "strike_hi", "end_date", "volume"]
    ].sort_values(["bet_type", "strike_lo"])
)

print(f"\nSaved full metadata to: {meta_path}")
print(f"Saved main-sample metadata to: {main_meta_path}")

Full metadata sample:
  Markets: 3,714

Main metadata sample:
  Markets: 3,700
  Excluded markets: 14

Excluded markets by bet type:
bet_type
dip      12
reach     2


,market_id,bet_type,question,strike_lo,strike_hi,end_date,volume
2412,516873,dip,"Will Bitcoin dip to $20,000 by December 31, 2025?",20000.0,NaN,2025-12-31 12:00:00+00:00,4.003570e+06
3170,1473090,dip,"Will Bitcoin dip to $20,000 in March?",20000.0,NaN,2026-04-01 04:00:00+00:00,3.150011e+06
3367,1823788,dip,"Will Bitcoin dip to $20,000 in April?",20000.0,NaN,2026-05-01 04:00:00+00:00,1.818739e+06
3171,1473087,dip,"Will Bitcoin dip to $25,000 in March?",25000.0,NaN,2026-04-01 04:00:00+00:00,1.314840e+06
3368,1823787,dip,"Will Bitcoin dip to $25,000 in April?",25000.0,NaN,2026-05-01 04:00:00+00:00,3.961066e+05
3172,1473086,dip,"Will Bitcoin dip to $30,000 in March?",30000.0,NaN,2026-04-01 04:00:00+00:00,6.919259e+05
3369,1823786,dip,"Will Bitcoin dip to $30,000 in April?",30000.0,NaN,2026-05-01 04:00:00+00:00,8.325713e+05
3567,2132787,dip,"Will Bitcoin dip to $30,000 in May?",30000.0,NaN,2026-06-01 04:00:00+00:00,5.358629e+05
2959,1303410,dip,"Will Bitcoin dip to $35,000 in February?",35000.0,NaN,2026-03-01 05:00:00+00:00,4.546147e+06
3173,1473084,dip,"Will Bitcoin dip to $35,000 in March?",35000.0,NaN,2026-04-01 04:00:00+00:00,1.708798e+06



Saved full metadata to: /Users/giannandreadestefano/TESI FINANCE/Thesis Project/data/processed/polymarket_btc_meta.csv
Saved main-sample metadata to: /Users/giannandreadestefano/TESI FINANCE/Thesis Project/data/processed/polymarket_btc_meta_main.csv


In [47]:
# ============================================================
# Load or download Polymarket BTC price histories
# ============================================================

prices_path = PROCESSED_DIR / "polymarket_btc_prices.csv"
errors_path = PROCESSED_DIR / "polymarket_btc_price_errors.csv"

metadata_columns = [
    "market_id", "event_id", "bet_type", "question",
    "strike_lo", "strike_hi", "start_date", "end_date",
    "closed", "volume", "yes_token", "main_sample"
]

if DOWNLOAD_POLYMARKET:
    print("Downloading Polymarket BTC price histories...")
    print(f"Markets to process: {len(df_meta):,}")
    print(f"Output file: {prices_path}\n")

    price_frames = []
    error_records = []

    for i, row in df_meta.iterrows():
        df_history = get_history(row["yes_token"])

        if df_history.empty:
            error_records.append({
                "market_id": row["market_id"],
                "yes_token": row["yes_token"],
                "bet_type": row["bet_type"],
                "question": row["question"],
                "end_date": row["end_date"],
                "error": "empty_history",
            })
        else:
            for col in metadata_columns:
                df_history[col] = row[col]
            price_frames.append(df_history)

        if (i + 1) % 100 == 0 or (i + 1) == len(df_meta):
            print(f"Processed {i + 1:,}/{len(df_meta):,} markets")

        time.sleep(REQUEST_DELAY)

    if not price_frames:
        raise ValueError("No Polymarket price histories were downloaded.")

    df_prices = pd.concat(price_frames, ignore_index=True)
    df_price_errors = pd.DataFrame(error_records)

else:
    if not prices_path.exists():
        raise FileNotFoundError(
            f"Price history file not found: {prices_path}\n"
            "Set DOWNLOAD_DATA = True in config.py to download price histories."
        )

    df_prices = pd.read_csv(prices_path)
    df_price_errors = pd.read_csv(errors_path) if errors_path.exists() else pd.DataFrame()

    print(f"Loaded existing Polymarket BTC price histories from: {prices_path}")

# Basic cleaning applied both after download and after loading.
df_prices["datetime"] = pd.to_datetime(df_prices["datetime"], errors="coerce", utc=True)
df_prices["start_date"] = pd.to_datetime(df_prices["start_date"], errors="coerce", utc=True)
df_prices["end_date"] = pd.to_datetime(df_prices["end_date"], errors="coerce", utc=True)

df_prices["price"] = pd.to_numeric(df_prices["price"], errors="coerce")
df_prices["volume"] = pd.to_numeric(df_prices["volume"], errors="coerce").fillna(0)
df_prices["strike_lo"] = pd.to_numeric(df_prices["strike_lo"], errors="coerce")
df_prices["strike_hi"] = pd.to_numeric(df_prices["strike_hi"], errors="coerce")

df_prices = (
    df_prices
    .dropna(subset=["datetime", "price", "market_id", "yes_token"])
    .sort_values(["market_id", "datetime"])
    .reset_index(drop=True)
)

df_prices.to_csv(prices_path, index=False)
df_price_errors.to_csv(errors_path, index=False)

print("=" * 70)
print("Polymarket BTC price histories ready.")
print(f"Saved prices to: {prices_path}")
print(f"Saved errors to: {errors_path}")
print(f"Total observations: {len(df_prices):,}")
print(f"Markets with history: {df_prices['market_id'].nunique():,}")
print(f"Markets without history: {len(df_price_errors):,}")
print(f"Datetime range: {df_prices['datetime'].min()} -> {df_prices['datetime'].max()}")

Markets to process: 3,714
Output file: /Users/giannandreadestefano/TESI FINANCE/Thesis Project/data/processed/polymarket_btc_prices.csv

Processed 100/3,714 markets
Processed 200/3,714 markets
Processed 300/3,714 markets
Processed 400/3,714 markets
Processed 500/3,714 markets
Processed 600/3,714 markets
Processed 700/3,714 markets
Processed 800/3,714 markets
Processed 900/3,714 markets
Processed 1,000/3,714 markets
Processed 1,100/3,714 markets
Processed 1,200/3,714 markets
Processed 1,300/3,714 markets
Processed 1,400/3,714 markets
Processed 1,500/3,714 markets
Processed 1,600/3,714 markets
Processed 1,700/3,714 markets
Processed 1,800/3,714 markets
Processed 1,900/3,714 markets
Processed 2,000/3,714 markets
Processed 2,100/3,714 markets
Processed 2,200/3,714 markets
Processed 2,300/3,714 markets
Processed 2,400/3,714 markets
Processed 2,500/3,714 markets
Processed 2,600/3,714 markets
Processed 2,700/3,714 markets
Processed 2,800/3,714 markets
Processed 2,900/3,714 markets
Processed 3

In [49]:
# ============================================================
# Price history diagnostics
# ============================================================

print("=" * 70)
print("Polymarket BTC price history diagnostics")
print("=" * 70)

print(f"Total observations: {len(df_prices):,}")
print(f"Unique markets with prices: {df_prices['market_id'].nunique():,}")
print(f"Unique YES tokens with prices: {df_prices['yes_token'].nunique():,}")
print(f"Markets without history: {len(df_price_errors):,}")

print("\nDatetime range:")
print(f"  Min: {df_prices['datetime'].min()}")
print(f"  Max: {df_prices['datetime'].max()}")

print("\nPrice distribution:")
display(df_prices["price"].describe().to_frame("price").round(4))

print("\nObservations per market:")
obs_per_market = df_prices.groupby("market_id").size()
display(obs_per_market.describe().to_frame("observations_per_market").round(2))

print("\nMarkets with price history by bet type:")
display(
    df_prices.groupby("bet_type")["market_id"]
    .nunique()
    .to_frame("n_markets")
)

print("\nMain sample coverage:")
display(
    df_prices.groupby("main_sample")["market_id"]
    .nunique()
    .to_frame("n_markets")
)

Polymarket BTC price history diagnostics
Total observations: 31,596
Unique markets with prices: 3,630
Unique YES tokens with prices: 3,630
Markets without history: 84

Datetime range:
  Min: 2024-03-09 00:00:03+00:00
  Max: 2026-06-03 00:00:10+00:00

Price distribution:


,price
count,31596.0000
mean,0.2534
std,0.3173
min,0.0005
25%,0.0200
50%,0.1050
75%,0.3750
max,0.9995



Observations per market:


,observations_per_market
count,3630.00
mean,8.70
std,18.27
min,1.00
25%,7.00
50%,7.00
75%,7.00
max,368.00



Markets with price history by bet type:


,n_markets
bet_type,
above,1565
dip,353
range,1331
reach,381



Main sample coverage:


,n_markets
main_sample,
False,14
True,3616


In [51]:
# ============================================================
# Price quality checks
# ============================================================

invalid_prices = df_prices[
    (df_prices["price"] < 0) |
    (df_prices["price"] > 1)
].copy()

print(f"Invalid prices outside [0, 1]: {len(invalid_prices):,}")
if len(invalid_prices) > 0:
    display(invalid_prices.head(20))


few_obs_markets = obs_per_market[obs_per_market < 2]

print(f"\nMarkets with fewer than 2 price observations: {len(few_obs_markets):,}")
if len(few_obs_markets) > 0:
    display(
        df_prices[df_prices["market_id"].isin(few_obs_markets.index)]
        [["market_id", "bet_type", "question", "datetime", "price", "end_date", "main_sample"]]
        .head(30)
    )


duplicate_rows = df_prices.duplicated(
    subset=["market_id", "datetime"],
    keep=False
)

print(f"\nDuplicate market-datetime rows: {duplicate_rows.sum():,}")
if duplicate_rows.sum() > 0:
    display(
        df_prices.loc[
            duplicate_rows,
            ["market_id", "datetime", "price", "bet_type", "question"]
        ].head(30)
    )

Invalid prices outside [0, 1]: 0

Markets with fewer than 2 price observations: 184


,market_id,bet_type,question,datetime,price,end_date,main_sample
1472,1082789,reach,"Will Bitcoin reach $90,000 in January?",2026-01-02 00:00:12+00:00,0.8950,2026-02-01 05:00:00+00:00,True
3240,1303387,dip,"Will Bitcoin dip to $80,000 in February?",2026-02-01 00:00:20+00:00,0.9860,2026-03-01 05:00:00+00:00,True
3720,1317110,dip,"Will Bitcoin dip to $74,000 February 2-8?",2026-02-03 00:00:38+00:00,0.3200,2026-02-09 05:00:00+00:00,True
5563,1473069,reach,"Will Bitcoin reach $70,000 in March?",2026-03-02 00:00:52+00:00,0.6550,2026-04-01 04:00:00+00:00,True
6262,1731388,range,"Will the price of Bitcoin be between $60,000 a...",2026-03-27 00:00:56+00:00,0.0260,2026-04-02 16:00:00+00:00,True
6266,1731393,range,"Will the price of Bitcoin be between $64,000 a...",2026-03-27 00:00:55+00:00,0.1250,2026-04-02 16:00:00+00:00,True
6343,1743623,range,"Will the price of Bitcoin be between $58,000 a...",2026-03-28 00:00:56+00:00,0.0285,2026-04-03 16:00:00+00:00,True
6350,1743629,range,"Will the price of Bitcoin be between $64,000 a...",2026-03-28 00:00:55+00:00,0.1950,2026-04-03 16:00:00+00:00,True
6351,1743633,range,"Will the price of Bitcoin be between $66,000 a...",2026-03-28 00:00:55+00:00,0.1800,2026-04-03 16:00:00+00:00,True
6359,1743638,range,"Will the price of Bitcoin be between $70,000 a...",2026-03-28 00:00:58+00:00,0.1000,2026-04-03 16:00:00+00:00,True



Duplicate market-datetime rows: 0


In [55]:
# ============================================================
# Price-history analysis flags
# ============================================================

MIN_PRICE_OBS = 2

obs_per_market = df_prices.groupby("market_id").size()

df_prices["n_price_obs"] = df_prices["market_id"].map(obs_per_market)
df_prices["price_sample"] = df_prices["n_price_obs"] >= MIN_PRICE_OBS

df_prices["analysis_sample"] = (
    df_prices["main_sample"] &
    df_prices["price_sample"]
)

prices_main_path = PROCESSED_DIR / "polymarket_btc_prices_main.csv"
prices_analysis_path = PROCESSED_DIR / "polymarket_btc_prices_analysis.csv"

df_prices_main = df_prices[df_prices["main_sample"]].copy()
df_prices_analysis = df_prices[df_prices["analysis_sample"]].copy()

df_prices.to_csv(prices_path, index=False)
df_prices_main.to_csv(prices_main_path, index=False)
df_prices_analysis.to_csv(prices_analysis_path, index=False)

n_low_history_markets = (
    df_prices.loc[~df_prices["price_sample"], "market_id"]
    .nunique()
)

print("Full price-history sample:")
print(f"  Observations: {len(df_prices):,}")
print(f"  Markets: {df_prices['market_id'].nunique():,}")

print("\nMain price-history sample:")
print(f"  Observations: {len(df_prices_main):,}")
print(f"  Markets: {df_prices_main['market_id'].nunique():,}")

print("\nAnalysis price-history sample:")
print(f"  Observations: {len(df_prices_analysis):,}")
print(f"  Markets: {df_prices_analysis['market_id'].nunique():,}")

print("\nMarkets excluded by price_sample flag:")
print(f"  Markets with fewer than {MIN_PRICE_OBS} observations: {n_low_history_markets:,}")

print(f"\nSaved full prices to: {prices_path}")
print(f"Saved main prices to: {prices_main_path}")
print(f"Saved analysis prices to: {prices_analysis_path}")

Full price-history sample:
  Observations: 31,596
  Markets: 3,630

Main price-history sample:
  Observations: 30,244
  Markets: 3,616

Analysis price-history sample:
  Observations: 30,060
  Markets: 3,432

Markets excluded by price_sample flag:
  Markets with fewer than 2 observations: 184

Saved full prices to: /Users/giannandreadestefano/TESI FINANCE/Thesis Project/data/processed/polymarket_btc_prices.csv
Saved main prices to: /Users/giannandreadestefano/TESI FINANCE/Thesis Project/data/processed/polymarket_btc_prices_main.csv
Saved analysis prices to: /Users/giannandreadestefano/TESI FINANCE/Thesis Project/data/processed/polymarket_btc_prices_analysis.csv


In [59]:
# ============================================================
# Consistency checks between metadata and price history
# ============================================================

metadata_markets = set(df_meta["market_id"])
price_markets = set(df_prices["market_id"])

markets_without_prices = metadata_markets - price_markets
markets_with_prices_not_in_metadata = price_markets - metadata_markets

print("Consistency checks")
print("=" * 70)
print(f"Markets in metadata: {len(metadata_markets):,}")
print(f"Markets in price history: {len(price_markets):,}")
print(f"Metadata markets without price history: {len(markets_without_prices):,}")
print(f"Price-history markets not in metadata: {len(markets_with_prices_not_in_metadata):,}")

if len(markets_with_prices_not_in_metadata) > 0:
    print("\nWarning: some price-history markets are not present in metadata.")

Consistency checks
Markets in metadata: 3,714
Markets in price history: 3,630
Metadata markets without price history: 84
Price-history markets not in metadata: 0


In [57]:
# ============================================================
# Final Polymarket export summary
# ============================================================

print("=" * 70)
print("Final Polymarket export summary")
print("=" * 70)

print("Metadata files:")
print(f"  Full metadata: {meta_path}")
print(f"  Main metadata: {main_meta_path}")

print("\nPrice-history files:")
print(f"  Full prices: {prices_path}")
print(f"  Main prices: {prices_main_path}")
print(f"  Analysis prices: {prices_analysis_path}")
print(f"  Price errors: {errors_path}")

print("\nMetadata samples:")
print(f"  Full markets: {df_meta['market_id'].nunique():,}")
print(f"  Main markets: {df_meta_main['market_id'].nunique():,}")

print("\nPrice-history samples:")
print(f"  Full observations: {len(df_prices):,}")
print(f"  Full markets: {df_prices['market_id'].nunique():,}")

print(f"  Main observations: {len(df_prices_main):,}")
print(f"  Main markets: {df_prices_main['market_id'].nunique():,}")

print(f"  Analysis observations: {len(df_prices_analysis):,}")
print(f"  Analysis markets: {df_prices_analysis['market_id'].nunique():,}")

print("\nAnalysis sample by bet type:")
display(
    df_prices_analysis.groupby("bet_type")["market_id"]
    .nunique()
    .to_frame("n_markets")
)

print("\nAnalysis sample date range:")
print("  Min datetime:", df_prices_analysis["datetime"].min())
print("  Max datetime:", df_prices_analysis["datetime"].max())

print("\nFinal quality checks:")
print("  Invalid prices:", ((df_prices["price"] < 0) | (df_prices["price"] > 1)).sum())
print("  Duplicate market-datetime rows:", df_prices.duplicated(subset=["market_id", "datetime"]).sum())
print("  Missing price:", df_prices["price"].isna().sum())
print("  Missing datetime:", df_prices["datetime"].isna().sum())

Final Polymarket export summary
Metadata files:
  Full metadata: /Users/giannandreadestefano/TESI FINANCE/Thesis Project/data/processed/polymarket_btc_meta.csv
  Main metadata: /Users/giannandreadestefano/TESI FINANCE/Thesis Project/data/processed/polymarket_btc_meta_main.csv

Price-history files:
  Full prices: /Users/giannandreadestefano/TESI FINANCE/Thesis Project/data/processed/polymarket_btc_prices.csv
  Main prices: /Users/giannandreadestefano/TESI FINANCE/Thesis Project/data/processed/polymarket_btc_prices_main.csv
  Analysis prices: /Users/giannandreadestefano/TESI FINANCE/Thesis Project/data/processed/polymarket_btc_prices_analysis.csv
  Price errors: /Users/giannandreadestefano/TESI FINANCE/Thesis Project/data/processed/polymarket_btc_price_errors.csv

Metadata samples:
  Full markets: 3,714
  Main markets: 3,700

Price-history samples:
  Full observations: 31,596
  Full markets: 3,630
  Main observations: 30,244
  Main markets: 3,616
  Analysis observations: 30,060
  Analysi

,n_markets
bet_type,
above,1533
dip,274
range,1315
reach,310



Analysis sample date range:
  Min datetime: 2024-03-09 00:00:03+00:00
  Max datetime: 2026-06-03 00:00:10+00:00

Final quality checks:
  Invalid prices: 0
  Duplicate market-datetime rows: 0
  Missing price: 0
  Missing datetime: 0


In [61]:
# ============================================================
# Save final Polymarket sample summary table
# ============================================================

polymarket_summary = pd.DataFrame({
    "sample": [
        "Full metadata",
        "Main metadata",
        "Full price history",
        "Main price history",
        "Analysis price history",
    ],
    "definition": [
        "All cleaned BTC Polymarket metadata",
        "Full metadata with economically comparable strikes",
        "Full metadata with available YES-token price history",
        "Full price history restricted to main_sample markets",
        "Main price history with at least two price observations per market",
    ],
    "markets": [
        df_meta["market_id"].nunique(),
        df_meta_main["market_id"].nunique(),
        df_prices["market_id"].nunique(),
        df_prices_main["market_id"].nunique(),
        df_prices_analysis["market_id"].nunique(),
    ],
    "observations": [
        len(df_meta),
        len(df_meta_main),
        len(df_prices),
        len(df_prices_main),
        len(df_prices_analysis),
    ],
})

summary_path = TABLES_DIR / "polymarket_sample_summary.csv"
polymarket_summary.to_csv(summary_path, index=False)

display(polymarket_summary)

print(f"Saved Polymarket summary table to: {summary_path}")

,sample,definition,markets,observations
0,Full metadata,All cleaned BTC Polymarket metadata,3714,3714
1,Main metadata,Full metadata with economically comparable str...,3700,3700
2,Full price history,Full metadata with available YES-token price h...,3630,31596
3,Main price history,Full price history restricted to main_sample m...,3616,30244
4,Analysis price history,Main price history with at least two price obs...,3432,30060


Saved Polymarket summary table to: /Users/giannandreadestefano/TESI FINANCE/Thesis Project/outputs/tables/polymarket_sample_summary.csv


In [63]:
# ============================================================
# Save final Polymarket analysis dataset
# ============================================================

polymarket_final_path = FINAL_DIR / "polymarket_btc_prices_analysis.csv"

df_prices_analysis.to_csv(polymarket_final_path, index=False)

print(f"Saved final Polymarket analysis dataset to: {polymarket_final_path}")
print(f"Final Polymarket markets: {df_prices_analysis['market_id'].nunique():,}")
print(f"Final Polymarket observations: {len(df_prices_analysis):,}")

Saved final Polymarket analysis dataset to: /Users/giannandreadestefano/TESI FINANCE/Thesis Project/data/final/polymarket_btc_prices_analysis.csv
Final Polymarket markets: 3,432
Final Polymarket observations: 30,060
